# 🛡️ SAFER FDS: AI Engine v3
**Smart AI Fraud Evaluation & Risk Intelligence**

Notebook ini mendemonstrasikan pipeline end-to-end Machine Learning, Advanced Feature Engineering, dan Deteksi Overfitting/Underfitting v3 untuk **SAFER FDS** (Fraud Detection System). Notebook ini siap dipindahkan (copy-paste) langsung ke Kaggle maupun Google Colab.

### Arsitektur AI FDS
1. **Data Ingestion & Preprocessing**: Memuat 550.000 data transaksi sintetis Indonesia dan melakukan rekayasa fitur tingkat lanjut (siklikal jam dan rasio finansial).
2. **Risk Scoring Engine (XGBoost & LightGBM)**: Melatih model klasifikasi biner untuk mendeteksi transaksi Fraud vs Non-Fraud dengan rasio logis (10%).
3. **Advanced Generalization Analysis**: Mendeteksi status Overfitting/Underfitting secara kuantitatif serta memvisualisasikan gap performa latih vs uji.
4. **Fraud Pattern Breakdown**: Mengevaluasi performa model secara terperinci untuk masing-masing 8 skenario fraud spesifik perbankan Indonesia.
5. **Interactive Dashboard Visualizations**: Menyajikan chart evaluasi profesional (Heatmap Confusion Matrix, Kurva ROC & Precision-Recall, dan Top 15 Feature Importances).

In [ ]:
# 1. Install Dependencies di Lingkungan Kaggle
!pip install xgboost lightgbm scikit-learn pandas numpy joblib matplotlib seaborn

In [ ]:
# 2. Import Libraries
import os
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# ML libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, accuracy_score,
    roc_curve, precision_recall_curve, auc
)
import xgboost as xgb
import lightgbm as lgb

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

## 3. Dataset Path Configuration
Upload file `train_transactions.csv` dan `test_transactions.csv` sebagai dataset baru di Kaggle, kemudian sesuaikan path direktori input di bawah ini sesuai nama dataset yang kamu buat.

In [ ]:
# Tentukan path dataset input di Kaggle
DATASET_DIR = Path("/kaggle/input/safer-fds-v3-dataset")

TRAIN_PATH = DATASET_DIR / "train_transactions.csv"
TEST_PATH = DATASET_DIR / "test_transactions.csv"

# Cek ketersediaan file
if not TRAIN_PATH.exists():
    print("Mencari lokasi dataset secara otomatis...")
    input_dirs = list(Path("/kaggle/input").glob("**/*transactions.csv"))
    if input_dirs:
        DATASET_DIR = input_dirs[0].parent
        TRAIN_PATH = DATASET_DIR / "train_transactions.csv"
        TEST_PATH = DATASET_DIR / "test_transactions.csv"
        print(f"Dataset ditemukan di: {DATASET_DIR}")
    else:
        print("ERROR: File dataset tidak ditemukan! Pastikan Anda sudah menambahkan dataset ke notebook.")

print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")

In [ ]:
# 4. Load Dataset
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Data Training Loaded: {len(train_df):,} baris ({train_df['is_fraud'].sum():,} fraud)")
print(f"Data Testing Loaded: {len(test_df):,} baris ({test_df['is_fraud'].sum():,} fraud)")

## 5. Advanced Feature Engineering
Menambahkan fitur jam siklikal (sin/cos) dan metrik rasio penipuan.

In [ ]:
def add_advanced_features(df):
    timestamps = pd.to_datetime(df["timestamp"])
    hours = timestamps.dt.hour
    df["hour_sin"] = np.sin(2 * np.pi * hours / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * hours / 24.0)
    df["amount_to_age_ratio"] = df["amount"] / (df["account_age_days"] + 1.0)
    df["dist_to_velocity_ratio"] = df["geo_distance_km"] / (df["velocity_count"].astype(float) + 1.0)
    df["amount_to_distance_ratio"] = df["amount"] / (df["geo_distance_km"] + 1.0)
    return df

## 6. Preprocessing (Label Encoding & Scaling)

In [ ]:
MODEL_FEATURES = [
    "sender_bank", "sender_lat", "sender_lng", "receiver_bank", "receiver_lat", "receiver_lng",
    "amount", "payment_rail", "ewallet_provider", "merchant", "merchant_category", "channel",
    "device_type", "device_brand", "is_new_device", "account_age_days", "is_velocity_anomaly",
    "is_geo_mismatch", "is_off_hours", "is_high_value_for_rail", "is_suspicious_ip",
    "is_risky_merchant", "is_new_account", "has_failed_attempts", "is_device_mismatch",
    "is_sim_swap", "is_unusual_beneficiary", "velocity_count", "geo_distance_km",
    "hour_sin", "hour_cos", "amount_to_age_ratio", "dist_to_velocity_ratio", "amount_to_distance_ratio"
]

CATEGORICAL_COLS = [
    "sender_bank", "receiver_bank", "payment_rail", "ewallet_provider",
    "merchant", "merchant_category", "channel", "device_type", "device_brand"
]

SCALED_COLS = [
    "amount", "account_age_days", "velocity_count", "geo_distance_km",
    "hour_sin", "hour_cos", "amount_to_age_ratio", "dist_to_velocity_ratio", "amount_to_distance_ratio"
]

# Jalankan Feature Engineering
train_df = add_advanced_features(train_df)
test_df = add_advanced_features(test_df)

X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df["is_fraud"].copy()
X_test = test_df[MODEL_FEATURES].copy()
y_test = test_df["is_fraud"].copy()

# Encode categoricals
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    X_train[col] = X_train[col].fillna("None").astype(str)
    X_test[col] = X_test[col].fillna("None").astype(str)
    all_val = pd.concat([X_train[col], X_test[col]]).unique()
    le.fit(all_val)
    X_train[col] = le.transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    label_encoders[col] = le

# Scale numerics
scaler = StandardScaler()
X_train[SCALED_COLS] = scaler.fit_transform(X_train[SCALED_COLS])
X_test[SCALED_COLS] = scaler.transform(X_test[SCALED_COLS])

## 7. Melatih Model V3 Ensemble (XGBoost + LightGBM)

In [ ]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

print("Training LightGBM...")
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "learning_rate": 0.05,
    "n_estimators": 300,
    "is_unbalance": True,
    "random_state": 42,
    "verbose": -1
}
lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=300)

## 8. Evaluasi Metrik & Deteksi Overfitting/Underfitting
Membandingkan metrik set data training vs testing secara ketat untuk mendeteksi tanda-tanda overfitting/underfitting.

In [ ]:
# 1. Evaluasi pada Training Set
xgb_train_probs = xgb_model.predict_proba(X_train)[:, 1]
lgb_train_probs = lgb_model.predict(X_train.to_numpy())
ens_train_probs = (xgb_train_probs + lgb_train_probs) / 2.0
ens_train_preds = (ens_train_probs >= 0.5).astype(int)

train_acc = accuracy_score(y_train, ens_train_preds)
train_prec, train_rec, train_f1, _ = precision_recall_fscore_support(y_train, ens_train_preds, average="binary")
train_roc_auc = roc_auc_score(y_train, ens_train_probs)

# 2. Evaluasi pada Testing Set
xgb_test_probs = xgb_model.predict_proba(X_test)[:, 1]
lgb_test_probs = lgb_model.predict(X_test.to_numpy())
ens_test_probs = (xgb_test_probs + lgb_test_probs) / 2.0
ens_test_preds = (ens_test_probs >= 0.5).astype(int)

test_acc = accuracy_score(y_test, ens_test_preds)
test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(y_test, ens_test_preds, average="binary")
test_roc_auc = roc_auc_score(y_test, ens_test_probs)
test_pr_auc = average_precision_score(y_test, ens_test_probs)

print("============================================================")
print("     OVERFITTING / UNDERFITTING GENERALIZATION REPORT")
print("============================================================")
print(f"Metrics      | Training Set | Testing Set  | Gap (Train - Test)")
print(f"-------------|--------------|--------------|-------------------")
print(f"Accuracy     | {train_acc:.4%}    | {test_acc:.4%}    | {train_acc - test_acc:+.4%}")
print(f"Precision    | {train_prec:.4%}    | {test_prec:.4%}    | {train_prec - test_prec:+.4%}")
print(f"Recall       | {train_rec:.4%}    | {test_rec:.4%}    | {train_rec - test_rec:+.4%}")
print(f"F1-Score     | {train_f1:.4%}    | {test_f1:.4%}    | {train_f1 - test_f1:+.4%}")
print(f"ROC-AUC      | {train_roc_auc:.4%}    | {test_roc_auc:.4%}    | {train_roc_auc - test_roc_auc:+.4%}")
print("============================================================")

# Analisis Generalisasi
f1_gap = train_f1 - test_f1
if f1_gap > 0.02:
    print(f"⚠️ Analisis: Model berpotensi OVERFIT (F1 Gap = {f1_gap:.2%}). Performa di Train jauh lebih baik.")
elif train_f1 < 0.85:
    print("⚠️ Analisis: Model berpotensi UNDERFIT. Akurasi F1-Score pada train set rendah (<85%).")
else:
    print("✅ Analisis: Model memiliki Generalisasi Sempurna (Good Fit)! Performa Train & Test sangat konsisten.")

In [ ]:
# 3. Breakdown Evaluasi per Pola Fraud Spesifik
print("\n=== FRAUD SCENARIO EVALUATION BREAKDOWN (TESTING SET) ===")
test_patterns = test_df["fraud_pattern"].copy()
unique_patterns = [p for p in test_patterns.unique() if p not in ["Normal", "General Fraud"]]
breakdown_f1 = []
breakdown_names = []

for pattern in unique_patterns:
    idx = (test_patterns == "Normal") | (test_patterns == pattern)
    y_sub = y_test[idx]
    preds_sub = ens_test_preds[idx]
    prec_sub, rec_sub, f1_sub, _ = precision_recall_fscore_support(y_sub, preds_sub, average="binary", zero_division=0)
    print(f"{pattern} -> Precision: {prec_sub:.4%}, Recall: {rec_sub:.4%}, F1: {f1_sub:.4%}")
    breakdown_f1.append(f1_sub)
    breakdown_names.append(pattern)

## 9. Visualisasi Performa Model & Analisis (Profesional Charts)
Membuat grafik matriks kebingungan, kurva ROC/PR, grafik overfitting, dan performa per skenario.

In [ ]:
# Set style untuk visualisasi bertema dark premium
sns.set_theme(style="darkgrid")
plt.rcParams["figure.facecolor"] = "#0d1117"
plt.rcParams["axes.facecolor"] = "#161b22"
plt.rcParams["text.color"] = "#c9d1d9"
plt.rcParams["axes.labelcolor"] = "#c9d1d9"
plt.rcParams["xtick.color"] = "#c9d1d9"
plt.rcParams["ytick.color"] = "#c9d1d9"

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, ens_test_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", ax=axes[0, 0], 
            xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"], cbar=False)
axes[0, 0].set_title("Ensemble Confusion Matrix Heatmap", fontsize=14, fontweight="bold", pad=10)
axes[0, 0].set_ylabel("Actual Label")
axes[0, 0].set_xlabel("Predicted Label")

# 2. ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, ens_test_probs)
axes[0, 1].plot(fpr, tpr, color="#6366f1", lw=3, label=f"Ensemble ROC (AUC = {test_roc_auc:.4f})")
axes[0, 1].plot([0, 1], [0, 1], color="#8c95a5", linestyle="--", lw=1.5)
axes[0, 1].set_xlim([0.0, 1.0])
axes[0, 1].set_ylim([0.0, 1.05])
axes[0, 1].set_xlabel("False Positive Rate")
axes[0, 1].set_ylabel("True Positive Rate")
axes[0, 1].set_title("Receiver Operating Characteristic (ROC) Curve", fontsize=14, fontweight="bold", pad=10)
axes[0, 1].legend(loc="lower right")

# 3. Precision-Recall Curve
prec_c, rec_c, _ = precision_recall_curve(y_test, ens_test_probs)
axes[1, 0].plot(rec_c, prec_c, color="#10b981", lw=3, label=f"Ensemble PR (AUC = {test_pr_auc:.4f})")
axes[1, 0].set_xlim([0.0, 1.0])
axes[1, 0].set_ylim([0.0, 1.05])
axes[1, 0].set_xlabel("Recall")
axes[1, 0].set_ylabel("Precision")
axes[1, 0].set_title("Precision-Recall Curve", fontsize=14, fontweight="bold", pad=10)
axes[1, 0].legend(loc="lower left")

# 4. Fraud Pattern Performance comparison
y_pos = np.arange(len(breakdown_names))
axes[1, 1].barh(y_pos, breakdown_f1, color="#f43f5e", height=0.6)
axes[1, 1].set_yticks(y_pos)
axes[1, 1].set_yticklabels(breakdown_names)
axes[1, 1].invert_yaxis()  # top-down
axes[1, 1].set_xlabel("F1-Score")
axes[1, 1].set_title("Performance (F1) by Fraud Skenario", fontsize=14, fontweight="bold", pad=10)
axes[1, 1].set_xlim([0.90, 1.01])

plt.tight_layout()
plt.savefig("model_v3_evaluation_plots.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Visualisasi Overfitting/Underfitting (Generalization Gap Chart)
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
train_vals = [train_acc, train_prec, train_rec, train_f1, train_roc_auc]
test_vals = [test_acc, test_prec, test_rec, test_f1, test_roc_auc]

x = np.arange(len(metrics_names))
width = 0.35

plt.figure(figsize=(10, 6), facecolor="#0d1117")
ax = plt.axes(facecolor="#161b22")
ax.bar(x - width/2, train_vals, width, label='Train Set (Generalisasi)', color='#3b82f6')
ax.bar(x + width/2, test_vals, width, label='Test Set (Evaluasi)', color='#10b981')

ax.set_ylabel('Scores')
ax.set_title('Generalization Analysis: Train vs Test Metrics (Ensemble V3)', fontsize=14, fontweight='bold', pad=10)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylim([0.90, 1.02])
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig("model_v3_generalization_chart.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 5. Feature Importance Plot (Top 15)
importances = xgb_model.feature_importances_
indices = np.argsort(importances)[::-1]
top_k = 15

plt.figure(figsize=(10, 6), facecolor="#0d1117")
ax = plt.axes(facecolor="#161b22")
plt.barh(range(top_k), importances[indices[:top_k]], color="#8b5cf6", align="center", height=0.6)
plt.yticks(range(top_k), [MODEL_FEATURES[i] for i in indices[:top_k]])
plt.gca().invert_yaxis()
plt.xlabel("Relative Importance")
plt.title("Top 15 Feature Importances (XGBoost V3)", fontsize=14, fontweight="bold", pad=10)
plt.tight_layout()
plt.savefig("model_v3_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Export Artifacts V3
Menyimpan bobot model ke file keluaran (/kaggle/working/)

In [ ]:
xgb_model.save_model("xgb_model_v3.json")
lgb_model.save_model("lgb_model_v3.txt")
joblib.dump(label_encoders, "label_encoders_v3.pkl")
joblib.dump(scaler, "scaler_v3.pkl")

print("Bobot Model V3 sukses disimpan di output directory!")